## Criação da base de dados principal com informações das estações de monitoramento no Brasil (MQAr_BR)

##### Bibliotecas

In [ ]:
import os, time, math, requests, pandas as pd
from datetime import datetime, timedelta, timezone
import pandas as pd
from collections import defaultdict
import re
import numpy as np
from pathlib import Path
import difflib
import unicodedata
from typing import Callable, Any, Optional
from functools import partial
from pandas.api.types import is_object_dtype, is_string_dtype
from collections import Counter
import unicodedata as ud

In [ ]:
import scripts.createUfStations_functions as cuf
import scripts

##### Dicionários e listas 

In [ ]:
# Colunas que precisam conter nas files de dados de ESTAÇÃO para cada estado
mqar_campos = [
        "UF","ID_OEMA","CIDADE","ID_MMA","ID_MMA_COMPLETO","POLUENTE","COD_POLUENTE",
        "CD_MUN","COD_UF_IBGE","PROPRIETARIO","PROP_ENTIDADE","OPERADOR","OP_ENTIDADE",
        "LATITUDE","LONGITUDE","MOBILIDADE","CATEGORIA","FUNCIONAMENTO","METODO",
        "MARCA",'INICIO', 'FIM',"FINALIDADE","MONITORAR","FONTE","CALIBRACAO","REALOCACAO",
        "OBS_CALIBRACAO","DADOS_MONITORAMENTO","RECONHECIDA","OBS_GERAIS",
        "STATUS","CERTIFICACAO","REP_ESPACIAL_DECLARADA"
    ]

In [ ]:
name_to_uf = {
    "acre":"AC","alagoas":"AL","amapa":"AP","amazonas":"AM","bahia":"BA","ceara":"CE",
    "distrito federal":"DF","espirito santo":"ES","goias":"GO","maranhao":"MA",
    "mato grosso":"MT","mato grosso do sul":"MS","minas gerais":"MG","para":"PA",
    "paraiba":"PB","parana":"PR","pernambuco":"PE","piaui":"PI","rio de janeiro":"RJ",
    "rio grande do norte":"RN","rio grande do sul":"RS","rondonia":"RO","roraima":"RR",
    "santa catarina":"SC","sao paulo":"SP","sergipe":"SE","tocantins":"TO"
}

In [ ]:
UF_TO_IBGE = {
    "AC":12,"AL":27,"AP":16,"AM":13,"BA":29,"CE":23,"DF":53,"ES":32,"GO":52,"MA":21,
    "MT":51,"MS":50,"MG":31,"PA":15,"PB":25,"PR":41,"PE":26,"PI":22,"RJ":33,"RN":24,
    "RS":43,"RO":11,"RR":14,"SC":42,"SP":35,"SE":28,"TO":17
}

def sigla_to_ibge(uf): return UF_TO_IBGE[uf.upper()]

In [ ]:
# Importar planilha com os códigos de poluentes
base = Path.cwd().parent  
out_dir = base / "data" / "dicionarios" 
out_dir.mkdir(parents=True, exist_ok=True)

df_cod = pd.read_csv(out_dir / 'CODIGO_POLUENTES.csv')

In [ ]:
# Importar planilha com as respostas do formulário das UFs
base = Path.cwd().parent  
fr_dir = base / "data" 
fr_dir.mkdir(parents=True, exist_ok=True)

forms = pd.read_csv(fr_dir / '2025_Formulário_Coleta_Respostas_UFs.csv')

#Indice das colunas com respostas sobre rede de monitoramento
# for i, c in enumerate(forms.columns):
#    print(f"[{i}] {c}")

idxs = [6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22] 

#### Passo a passo para criação da planilha MQAr completa:

__Sequência de ações para criar planilha final completa:__

1. Fazer upload de todas as planilhas UF_estacoes e conferir a quantidade de estações por planilha
2. Limpar e padronizar caracteres
3. Unir DFs e substituir categorias com escrita errada e padronizar (ex. FUNCIONAMENTO - Sim = Ativa)
4. Conferir nomes de poluentes e substituir pelo dicionário quando diferente (ex. PM10 = MP10)
5. Conferir linhas repetidas ou informações diferentes
6. Comparar planilha final com planilha PurpleAir e MQAr do ano anterior
7. Criar ID_MMA_COMPLETO
8. Explodir poluentes - um poluente por linha
9. Salvar e exportar

1. Fazer upload de todas as planilhas UF_estacoes e conferir a quantidade de estações por planilha

In [ ]:
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_ESTACOES" 
df_dir.mkdir(parents=True, exist_ok=True)
ufs_dfs = cuf.load_csvs(df_dir, prefix=None, recursive=False, limit=None)

In [ ]:
for name, d in ufs_dfs.items():
    print(f"\n=== {name} ===")
    display(d)

#### ID_MMA intermediário

In [ ]:
def _ascii_lower(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    return s.lower()


def _normalize_dt_mixed(series: pd.Series, anchor="start") -> pd.Series:
    s = pd.to_datetime(series, errors="coerce", utc=True)
    raw = series.astype("string").str.strip().str.replace(r"[\/\.]", "-", regex=True)

    # YYYY-MM
    mask_ym = raw.str.match(r"^\d{4}-\d{1,2}$", na=False)
    if mask_ym.any():
        base = pd.to_datetime(raw[mask_ym] + "-01", errors="coerce", utc=True)
        s.loc[mask_ym] = base if anchor == "start" else base + pd.offsets.MonthEnd(0)

    # YYYY
    mask_y = raw.str.match(r"^\d{4}$", na=False)
    if mask_y.any():
        s.loc[mask_y] = pd.to_datetime(
            raw[mask_y] + ("-01-01" if anchor == "start" else "-12-31"),
            errors="coerce",
            utc=True,
        )

    return s.dt.tz_convert(None)


def _parse_existing_nums(id_series: pd.Series, uf: str) -> pd.Series:
    """Extrai os 4 dígitos finais dos IDs válidos daquela UF."""
    pat = f"^{uf}\\d{{4}}$"
    m = id_series.astype("string").str.fullmatch(pat, na=False)
    nums = id_series.where(m).str[-4:].astype("Int64", errors="ignore")
    return nums


def assign_id_mma_all_rules(
    df: pd.DataFrame,
    uf_col="UF",
    start_col="INICIO",
    station_col="ID_OEMA",
    id_col="ID_MMA",
    anchor="start",
    pr_fixed_order=None,  # lista em ordem das estações PR com IDs fixos
    blocked_ufs=("RJ", "ES", "SC", "PE", "PB", "MA", "CE", "BA"),  # não criar nem alterar
):
    """
    Regras:
    1) Em cada UF, ordenar por INICIO asc. Empate ou sem data por nome A>Z.
    2) SP: manter existentes e preencher só vazios, continuando após o maior existente.
    3) DF: mesmo de SP.
    4) PR: manter existentes e os da lista fixa; criar novos após o maior existente.
    5) UFs em blocked_ufs: não criar nem alterar.
    6) Outras UFs: preencher só vazios, iniciando de 0001 ou após o maior existente.
    """
    out = df.copy()

    for c in [uf_col, station_col]:
        out[c] = out.get(c, pd.Series(pd.NA, index=out.index)).astype("string")
    if id_col not in out.columns:
        out[id_col] = pd.Series(pd.NA, index=out.index, dtype="string")
    else:
        out[id_col] = out[id_col].astype("string")

    out[start_col] = _normalize_dt_mixed(out[start_col], anchor=anchor)
    name_key = out[station_col].map(_ascii_lower)
    start_key = out[start_col].fillna(pd.Timestamp.max)
    out = (
        out.assign(_name_key=name_key, _start_key=start_key)
        .sort_values([uf_col, "_start_key", "_name_key"], kind="mergesort", na_position="last")
        .reset_index(drop=True)
    )

    pr_fixed_order = pr_fixed_order or []
    pr_fixed_set = set(pr_fixed_order)

    result = []
    for uf, g in out.groupby(uf_col, sort=False, dropna=False):
        if not isinstance(uf, str) or uf.strip() == "":
            result.append(g)
            continue

        is_blocked = uf in blocked_ufs
        is_sp = uf == "SP"
        is_df = uf == "DF"
        is_pr = uf == "PR"

        gg = g.copy()
        existing_nums = _parse_existing_nums(gg[id_col], uf)
        max_existing = int(existing_nums.max()) if not existing_nums.dropna().empty else 0

        if is_pr and pr_fixed_set:
            is_fixed_station = gg[station_col].isin(pr_fixed_set)
        else:
            is_fixed_station = pd.Series(False, index=gg.index)

        if is_blocked:
            can_fill = pd.Series(False, index=gg.index)
        else:
            can_fill = gg[id_col].isna() | gg[id_col].str.strip().eq("")
            if is_pr and pr_fixed_set:
                can_fill = can_fill & ~is_fixed_station

        idx_to_fill = gg.index[can_fill]
        if len(idx_to_fill) > 0:
            start_n = max_existing + 1
            seq = pd.Series(range(start_n, start_n + len(idx_to_fill)), index=idx_to_fill)
            new_ids = uf + seq.astype(int).astype(str).str.zfill(4)

            while gg[id_col].isin(new_ids).any():
                start_n += 1
                seq = pd.Series(range(start_n, start_n + len(idx_to_fill)), index=idx_to_fill)
                new_ids = uf + seq.astype(int).astype(str).str.zfill(4)

            gg.loc[idx_to_fill, id_col] = new_ids

        result.append(gg)

    out2 = pd.concat(result, axis=0).sort_index()
    return out2.drop(columns=["_name_key", "_start_key"], errors="ignore")

In [ ]:
pr_ordem_fixa = [
    "CIC","STC","ASS","BOQ","CSN","PAR","UEG","RPR","SIX","CAS",
    "PGA","LON","MRGA","FOZ","CVEL"
]

blocked = ("RJ","ES","SC","PE","PB","MA","CE","BA")  # manter como está

for uf, d in ufs_dfs.items():
    ufs_dfs[uf] = assign_id_mma_all_rules(
        d,
        uf_col="UF",
        start_col="INICIO",
        station_col="ID_OEMA",
        id_col="ID_MMA",
        anchor="start",
        pr_fixed_order=pr_ordem_fixa,
        blocked_ufs=blocked
    )

pr_fixed_map = {
    "CIC":"PR0001","STC":"PR0002","ASS":"PR0003","BOQ":"PR0004","CSN":"PR0005",
    "PAR":"PR0006","UEG":"PR0007","RPR":"PR0008","SIX":"PR0009","CAS":"PR0010",
    "PGA":"PR0011","LON":"PR0012","MRGA":"PR0013","FOZ":"PR0014","CVEL":"PR0015"
}

def aplicar_map_pr_inplace(df, uf_col="UF", nome_col="ID_OEMA", id_col="ID_MMA"):
    if id_col not in df.columns:
        df[id_col] = pd.NA
    mask = (df[uf_col].astype(str).str.upper().eq("PR")) & (df[id_col].isna() | df[id_col].astype(str).str.strip().eq(""))
    df.loc[mask, id_col] = df.loc[mask, nome_col].map(pr_fixed_map)

for name, d in ufs_dfs.items():
    aplicar_map_pr_inplace(d, uf_col="UF", nome_col="ID_OEMA", id_col="ID_MMA")
    print(f"\n=== {name} ===")
    display(d)

2. Limpar e padronizar caracteres

In [ ]:
# for name, d in ufs_dfs.copy().items():
#     dfs_clean = cuf.clean_df_all_text(d, in_place=True)

3. Unir DFs 

In [ ]:
uf_frame, uf_conflicts = cuf.merge_by_id_multi(
    ufs_dfs.copy(),
    id_cols=['ID_OEMA','POLUENTE'],
    source_priority=None,
    drop_empty_strings=True,
)

In [ ]:
uf_frame

4. Conferir nomes de poluentes e substituir pelo dicionário quando diferente (ex. PM10 = MP10)

In [ ]:
def _erase(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _norm(s: str) -> str:
    s = str(s)
    s = re.sub(r"\(.*?\)|\[.*?\]", "", s).replace("µ", "u")
    s = _erase(s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def build_cod_map(df_cod, col_sinonimos=None):
    m = {}
    for _, r in df_cod.iterrows():
        m[_norm(r["POLUENTE"])] = r["NOME_PASTA"]
        if col_sinonimos and col_sinonimos in df_cod.columns and pd.notna(r[col_sinonimos]):
            for alt in str(r[col_sinonimos]).split("|"):
                alt = alt.strip()
                if alt:
                    m[_norm(alt)] = r["NOME_PASTA"]
    return m

def normalize_pols_cell(text, cod_map, sep=r"[;,/|]+"):
    if pd.isna(text):
        return ""
    parts = re.split(sep, str(text))
    seen, out = set(), []
    for p in parts:
        k = _norm(p)
        name = cod_map.get(k)
        if name and name not in seen:
            seen.add(name); out.append(name)
    return ",".join(sorted(out))

# uso linha a linha
cod_map = build_cod_map(df_cod)  # faça uma vez
uf_frame["POLUENTE"] = uf_frame["POLUENTE"].apply(lambda s: normalize_pols_cell(s, cod_map))

In [ ]:
def limpar_ipynb_poluente(df, col="POLUENTE"):
    s = df[col].astype(str)

    # remove tokens contendo ".ipynb_checkpoints"
    s = s.str.replace(r'(?i)(^|,)\s*[^,]*ipynb[_-]?checkpoints[^,]*(?=,|$)', '', regex=True)
    # remove tokens contendo ".ipynb"
    s = s.str.replace(r'(?i)(^|,)\s*[^,]*\.ipynb[^,]*(?=,|$)', '', regex=True)

    # compacta vírgulas e espaços
    s = s.str.replace(r'\s*,\s*', ',', regex=True)
    s = s.str.replace(r',+', ',', regex=True).str.strip(', ')

    # dedup dos itens mantendo a ordem
    def _dedup(v):
        if not v:
            return v
        parts = [p for p in v.split(',') if p]
        seen = set()
        out = []
        for p in parts:
            if p not in seen:
                seen.add(p)
                out.append(p)
        return ",".join(out)

    df[col] = s.map(_dedup)
    return df

In [ ]:
def _to_ascii_upper(s: str) -> str:
    # remove acentos e normaliza subscrito/sobrescrito para dígitos
    subs = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    sups = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.translate(subs).translate(sups).upper().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _map_token(t: str) -> list[str]:
    """Mapeia um token para 0..n nomes padronizados."""
    t0 = _to_ascii_upper(t)

    # descartes
    if t0 in {"", "NAN", "NA", "NONE", "NULL", "ODOR", "ODORES"}:
        return []

    # PM/MP normalização
    t1 = t0.replace("PM", "MP")  # PM10 -> MP10, PM2.5 -> MP2.5
    t1 = t1.replace("MP1O", "MP10")  # O no lugar de zero

    # MP2.5 variações -> MP25
    if re.fullmatch(r"MP2([.,/]?5)?|MP25|PM2([.,/]?5)?", t0):
        return ["MP25"]

    if t1 in {"MP10"}:
        return ["MP10"]
    if t1 in {"MP1"}:
        return ["MP1"]

    # abreviações comuns
    direct = {
        "NOX": "NOX",
        "NO": "NO",
        "NO2": "NO2",
        "O3": "O3",
        "SO2": "SO2",
        "CO": "CO",
        "CH4": "CH4",
        "PTS": "PTS",
        "VOC": "VOC",
        "HCT": "HCT",
        "HCNM": "HCNM",
        "ERT": "ERT",
        "FMC": "FMC",
        "H2S": "H2S",
        "BENZENO": "BENZENO",
        "TOLUENO": "TOLUENO",
        "ETILBENZENO": "ETILBENZENO",
        "XILENO": "XILENO",
        "OXILENO": "OXILENO",
        "MPXILENO": "MPXILENO",
        "ACETAL": "ACETAL",
        "FORMAL": "FORMAL",
        "CH2O": "FORMAL",   # formaldeído
        "BEN": "BENZENO",
        "BENZ": "BENZENO",
        "TOL": "TOLUENO",
        "ETBEN": "ETILBENZENO",
        "ETILBEN": "ETILBENZENO",
        "MPXIL": "MPXILENO",
        "OXIL": "OXILENO",
        "XIL": "XILENO",
    }
    if t1 in direct:
        return [direct[t1]]

    # caso especial “ETIL,ORTO” já será quebrado pelo split; mas se vier colado:
    if t1 in {"ETIL,ORTO", "ETIL ORTO"}:
        return ["ETILBENZENO", "OXILENO"]

    # fallback: mantemos o token bruto normalizado
    return [t1]

def normalizar_coluna_poluente(df: pd.DataFrame, df_cod: pd.DataFrame, col="POLUENTE") -> pd.DataFrame:
    if col not in df.columns:
        return df.copy()

    # vocabulário permitido
    allowed = set(df_cod["NOME_PASTA"].astype(str).str.upper().str.strip())

    # separadores: vírgula, ;, /, |, e “ e ”
    sep_re = re.compile(r"\s*(?:,|;|/|\||\se\s)\s*", flags=re.I)

    def process_cell(val) -> str:
        if pd.isna(val):
            return ""
        tokens = []
        seen = set()
        # quebra
        parts = [p for p in sep_re.split(str(val)) if p.strip()]
        for p in parts:
            mapped = _map_token(p)
            for m in mapped:
                if m in allowed and m not in seen:
                    seen.add(m)
                    tokens.append(m)
        return ",".join(tokens)

    out = df.copy()
    out[col] = out[col].apply(process_cell)
    return out

def listar_tokens_restantes(df: pd.DataFrame, col="POLUENTE") -> list[str]:
    parts = df[col].astype(str).str.split(",").explode().dropna().str.strip()
    return sorted([t for t in parts.unique() if t])

# -------------------- uso --------------------
# df_cod precisa ter a coluna NOME_PASTA com os nomes oficiais
# df_merged é o seu DF com a coluna POLUENTE

uf_frame = limpar_ipynb_poluente(uf_frame, col="POLUENTE")
uf_frame = normalizar_coluna_poluente(uf_frame, df_cod, col="POLUENTE")
print(listar_tokens_restantes(uf_frame, "POLUENTE"))

5. Conferir linhas repetidas ou informações diferentes

In [ ]:
# 1) Linhas em que ID_OEMA se repetem
dups = uf_frame[uf_frame["ID_OEMA"].duplicated(keep=False)].sort_values("ID_OEMA")
print(dups)

# 2) Todos os IDs duplicados
dup_ids = uf_frame["ID_OEMA"][uf_frame["ID_OEMA"].duplicated()].unique()
print("Duplicated IDs:", dup_ids)

# 3) Quantas IDs foram repetidas
counts = uf_frame["ID_OEMA"].value_counts()
print(counts[counts > 1])

##### ID_MMA_COMPLETO intermediário

In [ ]:
df_full = explode_and_apply_PolCod(uf_frame, df_cod)

In [ ]:
from typing import Optional

def _str_no_dotzero(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

def _cod_poluente_clean(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s_num = pd.to_numeric(s, errors="coerce")
    out = _str_no_dotzero(s)
    m = s_num.notna()
    out.loc[m] = s_num.loc[m].astype("Int64").astype(str).str.zfill(3)
    return out.fillna("")

def _extract_uf_num(idmma: pd.Series) -> pd.DataFrame:
    s = idmma.astype("string").str.strip()
    uf = s.str.extract(r"^([A-Z]{2})", expand=False)
    num = s.str.extract(r"^[A-Z]{2}\s*0*?(\d+)$", expand=False)
    num = pd.to_numeric(num, errors="coerce").astype("Int64")
    return pd.DataFrame({"_UF": uf, "_NUM": num}, index=s.index)

# >>> ANOTAÇÃO CORRIGIDA AQUI <<<
def _pair_for_row(uf: Optional[str], num: Optional[int]) -> str:
    if uf is None or pd.isna(uf):
        return ""
    uf = str(uf).upper()

    fixed_default = {
        "SP": "RA", "ES": "RA", "MG": "RA", "SC": "RA", "RS": "RA",
        "PR": "RA", "BA": "ND", "MA": "RA", "MT": "IA", "PE": "ND",
        "RR": "RS", "PB": "ND", "CE": "ND",
    }

    def ge(n: Optional[int], cutoff: int) -> bool:
        return n is not None and not pd.isna(n) and int(n) >= cutoff

    if uf == "RJ":
        return "ND" if ge(num, 1000) else "RA"
    if uf == "PA":
        return "ND" if ge(num, 1001) else "RA"
    if uf == "RN":
        return "ND" if ge(num, 1001) else "RA"

    return fixed_default.get(uf, "RA")

def make_id_mma_completo_fixos(df: pd.DataFrame) -> pd.DataFrame:
    out   = df.copy()
    idmma = _str_no_dotzero(out.get("ID_MMA", pd.Series(index=out.index)))
    cod   = _cod_poluente_clean(out.get("COD_POLUENTE", ""))

    meta = _extract_uf_num(idmma)
    uf, num = meta["_UF"], meta["_NUM"]

    pair = pd.Series((_pair_for_row(u, int(n) if not pd.isna(n) else None) for u, n in zip(uf, num)),
                     index=out.index, dtype="string")

    out["ID_MMA_COMPLETO"] = (idmma.fillna("") + pair.fillna("") + cod).astype("string")
    return out

In [ ]:
df_full = make_id_mma_completo_fixos(df_full)

6. Comparar planilha final com planilha PurpleAir

In [ ]:
# Importar planilha com dados do PurpleAir coletados internamente 
base = Path.cwd().parent  
out_dir = base / "data" / "DADOS_ESTACOES" / "Indicativas" 
out_dir.mkdir(parents=True, exist_ok=True)

df_purple = pd.read_csv(out_dir / 'Compiled_PurpleAirStations.csv')

In [ ]:
df_purple

In [ ]:
df_purple_full = explode_and_apply_PolCod(df_purple, df_cod)
df_purple_full = df_purple_full[df_purple_full["POLUENTE"] == "MP25"].copy()

In [ ]:
df_mma = df_full.copy()

In [ ]:
# LATITUDE
df_mma['LATITUDE'] = (
    pd.to_numeric(df_mma['LATITUDE'].astype(str).str.replace(',', '.'), errors='coerce'))
# LONGITUDE
df_mma['LONGITUDE'] = (
    pd.to_numeric(df_mma['LONGITUDE'].astype(str).str.replace(',', '.'), errors='coerce'))

In [ ]:
def _collapse_dupe_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Coalesce valores entre colunas duplicadas com o mesmo nome (esq→dir) e retorna colunas únicas."""
    df = df.copy()
    uniques = pd.Index(df.columns).unique()
    out = pd.DataFrame(index=df.index)
    for c in uniques:
        block = df.loc[:, df.columns == c]
        out[c] = block.bfill(axis=1).iloc[:, 0]  # primeiro não-nulo
    return out

def _norm_basic(s: pd.Series) -> pd.Series:
    """Upper, trim, string; vazio vira ''."""
    return s.astype("string").fillna("").str.strip().str.upper()

def _norm_name(s: pd.Series) -> pd.Series:
    """
    Normalização forte para nomes:
    - strip, upper
    - remove acentos
    - troca pontuação por espaço
    - colapsa múltiplos espaços
    """
    t = s.astype("string").fillna("").str.strip().str.upper()
    # remove acentos
    t = t.map(lambda x: "".join(ch for ch in ud.normalize("NFKD", x) if not ud.combining(ch)))
    # troca pontuação por espaço
    t = t.str.replace(r"[^A-Z0-9]+", " ", regex=True)
    # colapsa espaços
    t = t.str.replace(r"\s+", " ", regex=True).str.strip()
    return t

def _nan_if_empty(s: pd.Series) -> pd.Series:
    """Converte '' (ou só espaços) em NA para coalescência correta."""
    s = s.astype("string")
    return s.mask(s.str.strip().eq(""))

# ---------------------------
# Merge principal
# ---------------------------

def merge_by_df(
    df_purple: pd.DataFrame,
    df_mma: pd.DataFrame,
    key_cols=("UF", "ID_OEMA", "POLUENTE"),
    mma_priority=True  # se True, df_mma prevalece quando ambos têm valor
) -> pd.DataFrame:
    # 1) tratar duplicadas
    p = _collapse_dupe_cols(df_purple)
    m = _collapse_dupe_cols(df_mma)

    # 2) chaves normalizadas para juntar
    #    - ID_OEMA: normalização forte (_norm_name)
    #    - UF, POLUENTE: normalização básica (_norm_basic)
    def _norm_for_key(colname: str, s: pd.Series) -> pd.Series:
        if colname.upper() == "ID_OEMA":
            return _norm_name(s)
        else:
            return _norm_basic(s)

    p_keys = [f"{k}__norm" for k in key_cols]
    m_keys = [f"{k}__norm" for k in key_cols]
    for k, pk, mk in zip(key_cols, p_keys, m_keys):
        p[pk] = _norm_for_key(k, p.get(k, pd.Series(index=p.index)))
        m[mk] = _norm_for_key(k, m.get(k, pd.Series(index=m.index)))

    # 3) descobrir colunas compartilhadas e exclusivas
    shared = sorted(set(p.columns).intersection(m.columns) - set(key_cols) - set(p_keys) - set(m_keys))
    only_p = [c for c in p.columns if c not in m.columns and c not in p_keys]
    only_m = [c for c in m.columns if c not in p.columns and c not in m_keys]

    # 4) outer merge nas chaves normalizadas
    merged = p.merge(
        m, left_on=p_keys, right_on=m_keys, how="outer",
        suffixes=("_purple", "_mma"), indicator=True
    )

    # 5) construir colunas finais
    out = pd.DataFrame(index=merged.index)

    # chaves finais: se houver conflito, priorizar MMA (mais confiável)
    for k in key_cols:
        mma_k = _nan_if_empty(merged.get(f"{k}_mma"))
        pur_k = _nan_if_empty(merged.get(f"{k}_purple"))
        out[k] = mma_k.combine_first(pur_k)

    # colunas compartilhadas
    for c in shared:
        mma_c = _nan_if_empty(merged[f"{c}_mma"])
        pur_c = _nan_if_empty(merged[f"{c}_purple"])
        out[c] = (mma_c.combine_first(pur_c)) if mma_priority else (pur_c.combine_first(mma_c))

    # colunas só do purple
    for c in only_p:
        if f"{c}_purple" in merged:
            out[c] = merged[f"{c}_purple"]
        elif c in merged:
            out[c] = merged[c]

    # colunas só do mma
    for c in only_m:
        if f"{c}_mma" in merged:
            out[c] = merged[f"{c}_mma"]
        elif c in merged:
            out[c] = merged[c]

    # 6) status da correspondência
    out["RECONHECIDA"] = merged["_merge"].map({
        "both": "Reconhecida",
        "left_only": "Nao reconhecida",
        "right_only": ""
    })

    # 7) ordem de colunas: todas do purple + extras do mma + RECONHECIDA
    out_cols = list(p.columns.drop(p_keys)) + [c for c in out.columns if c not in p.columns]
    out = out[[c for c in out_cols if c in out.columns]]

    return out

In [ ]:
print(len(df_purple_full))
print(len(df_mma))

In [ ]:
dfs_merged = merge_by_df(df_purple_full.copy(), df_mma.copy())
dfs_merged

7. Explodir poluentes e preencher coluna com códigos - um poluente por linha

In [ ]:
# def explode_pol_mtd_marca(df):
#     linhas = []

#     for _, row in df.iterrows():
#         # --- POLUENTES (with MP expansion) ---
#         poluentes = [p.strip() for p in str(row.get("POLUENTE", "")).split(",") if p.strip()]
#         poluentes_expandidos = []
#         for p in poluentes:
#             if p.upper() == "MP":
#                 poluentes_expandidos.extend(["MP10", "MP25", "PTS"])
#             else:
#                 poluentes_expandidos.append(p)

#         # Ensure string inputs
#         metodo_raw = "" if pd.isna(row.get("METODO", "")) else str(row.get("METODO", "")).strip()
#         marca_raw  = "" if pd.isna(row.get("MARCA", ""))  else str(row.get("MARCA", "")).strip()

#         # --- Parse METODO with optional per-pollutant mapping ---
#         metodos_dict = {}
#         if metodo_raw:
#             for item in metodo_raw.split(","):
#                 item = item.strip()
#                 match = re.search(r"\((.*?)\)", item)
#                 if match:
#                     pols = [pp.strip() for pp in match.group(1).split(",")]
#                     for p in pols:
#                         if p.upper() == "MP":
#                             for mp in ["MP10", "MP25", "PTS"]:
#                                 metodos_dict[mp] = item
#                         else:
#                             metodos_dict[p] = item

#         # If there is a single generic method (no commas and no parentheses), apply to all
#         metodo_default = ""
#         if metodo_raw and ("," not in metodo_raw) and (not re.search(r"\(.*\)", metodo_raw)):
#             metodo_default = metodo_raw

#         # --- Parse MARCA with optional per-pollutant mapping ---
#         marcas_dict = {}
#         if marca_raw:
#             for item in marca_raw.split(","):
#                 item = item.strip()
#                 match = re.search(r"\((.*?)\)", item)
#                 if match:
#                     pols = [pp.strip() for pp in match.group(1).split(",")]
#                     for p in pols:
#                         if p.upper() == "MP":
#                             for mp in ["MP10", "MP25", "PTS"]:
#                                 marcas_dict[mp] = item
#                         else:
#                             marcas_dict[p] = item

#         # If there is a single generic brand (no commas and no parentheses), apply to all
#         marca_default = ""
#         if marca_raw and ("," not in marca_raw) and (not re.search(r"\(.*\)", marca_raw)):
#             marca_default = marca_raw

#         # --- Build exploded rows ---
#         for pol in poluentes_expandidos:
#             nova = row.copy()
#             nova["POLUENTE"] = pol
#             nova["METODO"] = metodos_dict.get(pol, metodo_default)
#             nova["MARCA"] = marcas_dict.get(pol, marca_default)
#             linhas.append(nova)

#     return pd.DataFrame(linhas).reset_index(drop=True)

In [ ]:
# def explode_and_apply_PolCod(df, df_cod):
#     for col in ["POLUENTE", "COD_POLUENTE", "NOME_PASTA"]:
#         if col in df_cod.columns:
#             df_cod[col] = df_cod[col].astype("string").str.strip()
            
#     # --- Carregar dicionário de poluentes ---
#     mapa_codigos = dict(zip(df_cod["POLUENTE"].str.strip(), df_cod["COD_POLUENTE"].str.strip()))
#     mapa_nome = dict(zip(df_cod["POLUENTE"].str.strip(), df_cod["NOME_PASTA"].str.strip()))

#     # Explodir POLUENTE
#     if "POLUENTE" in df.columns:
#         df = explode_pol_mtd_marca(df)

#     # Preencher COD_POLUENTE
#     df["COD_POLUENTE"] = df.get("POLUENTE", "").map(mapa_codigos).fillna("").astype(str)
#     df["COD_POLUENTE"] = df["COD_POLUENTE"].apply(lambda x: x.zfill(3) if x.isdigit() else x)

#     # Substituir POLUENTE pelo NOME_PASTA do dicionário
#     df["POLUENTE"] = df.get("POLUENTE", "").map(mapa_nome).fillna(df.get("POLUENTE", ""))

#     return df

In [ ]:
# df_full = explode_and_apply_PolCod(dfs_merged, df_cod)

Nota: Como a base de dados do ano anterior possui os poluentes já explodidos por linha, fazer a comparação entre estações após explodir poluentes nas estações

In [ ]:
path = Path.cwd().parent / "data" / "DADOS_ESTACOES" / "Dados_2024" / "Monitoramento_QAr_BR.csv"

def load_csv_robust(p: Path) -> pd.DataFrame:
    # 1) Auto-detect delimiter, UTF-8
    try:
        return pd.read_csv(p, sep=None, engine="python", encoding="utf-8")
    except Exception:
        pass
    # 2) Auto-detect delimiter, Windows-1252
    try:
        return pd.read_csv(p, sep=None, engine="python", encoding="cp1252")
    except Exception:
        pass
    # 3) Semicolon + cp1252 (very common in BR CSVs)
    try:
        return pd.read_csv(p, sep=";", engine="python", encoding="cp1252")
    except Exception:
        pass
    # 4) Comma + cp1252, skipping bad lines
    return pd.read_csv(p, sep=",", engine="python", encoding="cp1252", on_bad_lines="skip")

# Optional: detect if the file is actually an Excel saved with .csv extension
def load_tabular(p: Path) -> pd.DataFrame:
    with open(p, "rb") as f:
        sig = f.read(4)
    if sig == b"PK\x03\x04":  # XLSX signature
        return pd.read_excel(p, engine="openpyxl")
    return load_csv_robust(p)

df_old = load_tabular(path)

In [ ]:
df_full = dfs_merged.copy() 

In [ ]:
print(len(df_old))
print(len(df_full))

In [ ]:
id_col = "ID_OEMA"

def norm(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.upper()

def overlap(df_full, df_old):
    # normalized ID columns
    mma_id = norm(df_full[id_col])
    pur_id = norm(df_old[id_col])
    
    # IDs present in both dataframes
    common_ids = pd.Index(mma_id.dropna()).intersection(pur_id.dropna())
    print("IDs present in both:", len(common_ids))
    
    # rows involved on each side
    mma_overlap = df_full[mma_id.isin(common_ids)]
    pur_overlap = df_old[pur_id.isin(common_ids)]
    #print("Rows in df_full with overlapping ID_OEMA:", len(mma_overlap))
    #print("Rows in df_purple_exp with overlapping ID_OEMA:", len(pur_overlap))
    
    # summary count per ID showing how many times it appears in each df
    both = pd.concat(
        [
            pd.DataFrame({id_col: mma_id, "_src": "mma"}),
            pd.DataFrame({id_col: pur_id, "_src": "purple"}),
        ],
        ignore_index=True,
    )
    summary = (
        both.dropna(subset=[id_col])
            .groupby([id_col, "_src"]).size()
            .unstack(fill_value=0)
            .query("mma > 0 and purple > 0")
            .sort_index()
    )
    print("Overlapping unique IDs:", summary.shape[0])

overlap(df_full, df_old)

In [ ]:
def fill_location_from_old(df_full, df_old):
    # ensure strings for matching
    df_full = df_full.copy()
    df_old = df_old.copy()
    df_full["ID_OEMA"] = df_full["ID_OEMA"].astype(str).str.strip()
    df_old["ID_OEMA"]  = df_old["ID_OEMA"].astype(str).str.strip()

    # subset of df_old with needed columns
    cols = ["ID_OEMA", "LONGITUDE", "LATITUDE", "CIDADE"]
    old_sub = df_old[cols].drop_duplicates(subset=["ID_OEMA"], keep="first")

    # merge to bring old info into full
    merged = df_full.merge(old_sub, on="ID_OEMA", how="left", suffixes=("", "_old"))

    # fill only where df_full is empty but df_old has data
    for col in ["LONGITUDE", "LATITUDE", "CIDADE"]:
        mask_full_empty = merged[col].isna() | merged[col].astype(str).str.strip().eq("")
        mask_old_nonempty = ~(merged[f"{col}_old"].isna() | merged[f"{col}_old"].astype(str).str.strip().eq(""))
        merged.loc[mask_full_empty & mask_old_nonempty, col] = merged.loc[
            mask_full_empty & mask_old_nonempty, f"{col}_old"
        ]

    # drop helper columns
    merged = merged.drop(columns=[c for c in merged.columns if c.endswith("_old")])
    return merged

In [ ]:
df_full = fill_location_from_old(df_full.copy(), df_old)

In [ ]:
df_full['LATITUDE'] = (
    pd.to_numeric(df_full['LATITUDE'].astype(str).str.replace(',', '.'), errors='coerce'))
# LONGITUDE
df_full['LONGITUDE'] = (
    pd.to_numeric(df_full['LONGITUDE'].astype(str).str.replace(',', '.'), errors='coerce'))

In [ ]:
print(len(df_full))
df_full['ID_OEMA'].nunique()

In [ ]:
# Conferir se ainda existem linhas duplicadas para o mesmo conjunto de flags

cols = ["ID_OEMA", "ID_MMA", "ID_MMA_COMPLETO", "UF"]

# 1) Flag duplicates by the key combo
dup_mask = df_full.duplicated(subset=cols, keep=False)  # marks all rows in each dup set

# 2) See only the duplicate rows
dups = df_full.loc[dup_mask].sort_values(cols)

# 3) Counts per duplicate key
dup_counts = (
    dups.groupby(cols, dropna=False)
        .size()
        .reset_index(name="COUNT")
        .sort_values("COUNT", ascending=False)
)
# dups
# dup_counts

In [ ]:
df_full = df_full.drop_duplicates(subset=cols, keep="first").copy()

8. Completar dados faltantes e padronizar respostas

In [ ]:
for col in df_full.columns:
    s = df_full[col]
    vc = s.dropna().value_counts()
    print(vc) 

In [ ]:
def replace_vals(df):
    # colunas alvo, só aplica se existirem
    cols = [c for c in ["CATEGORIA", "FUNCIONAMENTO", "STATUS", "RECONHECIDA", "MONITORAR", "FONTE", "CALIBRACAO" 
                       "PROP_ENTIDADE", "OP_ENTIDADE", "METODO", "MOBILIDADE", "FINALIDADE", "MONITORAR", 
                        "REALOCACAO", "REP_ESPACIAL_DECLARADA", "MARCA"] if c in df.columns]

    # regex para strings "vazias" comuns
    null_like = r'^\s*(na|n/a|none|null|nan|nat)?\s*$'

    # 1) normaliza vazios em todas as colunas alvo
    # for c in cols:
    #     df[c] = df[c].replace(null_like, pd.NA, regex=True).fillna("Nao declarado")

    # 2) mapeamentos específicos
    if "CATEGORIA" in df.columns:
        df["CATEGORIA"] = df["CATEGORIA"].replace({
            "N": "Nao declarado",
            "D": "Nao declarado",
            "Naodeclarado": "Nao declarado",
            "Meteorologica": "Nao declarado",
            "Meteorológica": "Nao declarado",
            "Nao Aplicavel": "Nao declarado",
            "CertificadaEPA": "Referencia",
            "Certificada EPA": "Referencia",
            "Equivalente": "Referencia",
            "Referência": "Referencia",
            "Não Declarada": "Nao declarado",
            "Não Aplicável": "Nao declarado"
        })

    if "PROP_ENTIDADE" in df.columns:
        df["PROP_ENTIDADE"] = df["PROP_ENTIDADE"].replace({
            "Público": "Publica",
            "Pública": "Publica",
            "Publico": "Publica"
        })

    if "OP_ENTIDADE" in df.columns:
        df["PROP_ENTIDADE"] = df["PROP_ENTIDADE"].replace({
            "-": "Nao declarado",
            "Pública": "Publica",
        })

    if "METODO" in df.columns:
        df["METODO"] = df["METODO"].replace({
            "-": "Nao declarado",
            "x": "Nao declarado",
        })

    if "FUNCIONAMENTO" in df.columns:
        df["FUNCIONAMENTO"] = df["FUNCIONAMENTO"].replace({
            "N": "Nao declarado",
            "D": "Nao declarado",
            "Autmatica": "Automatica",
            "Automatico": "Automatica",
            "Automático": "Automatica",
            "Não Declarada": "Nao declarado",
            "Automática": "Automatica",
            "Autmática": "Automatica"
        })

    if "STATUS" in df.columns:
        df["STATUS"] = df["STATUS"].replace({
            "Sim": "Ativa",
            "sim": "Ativa",
            "Não": "Inativa",
            "Nao": "Inativa",
            "nao": "Inativa",
            "NAO": "Inativa",
        })
        
    if "RECONHECIDA" in df.columns:
        df["RECONHECIDA"] = df["RECONHECIDA"].replace({
            "Nao declarado" : "Nao reconhecida", 
            "Não": "Nao reconhecida",
            "Nao": "Nao reconhecida",
            "nao": "Nao reconhecida",
            "NAO": "Nao reconhecida",
            "Nao reconhecido": "Nao reconhecida",
            "Naoreconhecida": "Nao reconhecida",
            "Sim": "Reconhecida"
        })

    if "REP_ESPACIAL_DECLARADA" in df.columns:
        df["REP_ESPACIAL_DECLARADA"] = df["REP_ESPACIAL_DECLARADA"].replace({
            "Não classificada" : "Nao declarado", 
            "Escala de Bairro": "Bairro",
            "Escala bairo": "Bairro",
            "Escala urbana": "Urbana",
            "Micro": "Microescala",
            "Media": "Mesoescala",
            "Média": "Mesoescala",
            "-": "Nao declarado",
            "Naoreconhecida": "Nao declarado"
        })

    if "FINALIDADE" in df.columns:
        df["FINALIDADE"] = df["FINALIDADE"].replace({
            "Licenciamento" : "Licenciamento Ambiental", 
            "Fornecer Dados": "Fornecer dados"
        })

    if "MONITORAR" in df.columns:
        df["FINALIDADE"] = df["FINALIDADE"].replace({
            "Não" : "Nao"
        })

    if "REALOCACAO" in df.columns:
        df["REALOCACAO"] = df["REALOCACAO"].replace({
            "Não" : "Nao"
        })

    if "MONITORAR" in df.columns:
        df["MONITORAR"] = df["MONITORAR"].replace({
            "Sm": "Sim"
        })

    if "MARCA" in df.columns:
        df["MARCA"] = df["MARCA"].replace({
            "x": "Nao declarado"
        })

    if "FONTE" in df.columns:
        df["FONTE"] = df["FONTE"].replace({
            "Consulta Interna": "Coleta interna",
            "Coleta Interna": "Coleta interna"
        })

    if "MOBILIDADE" in df.columns:
        df["MOBILIDADE"] = df["MOBILIDADE"].replace({
            "Móvel": "Movel"
        })

    if "CALIBRACAO" in df.columns:
        df["CALIBRACAO"] = df["CALIBRACAO"].replace({
            "diaria": "Diaria",
            "diária": "Diaria",
            "A cada 3 meses": "Trimestral",
            "A cada 1 mes": "Mensal",
            "A cada 1 mês": "Mensal",
            "Nao Aplicavel": "Nao aplicavel",
            "-": "Nao declarado",
            "x": "Nao declarado",
            "Não Aplicável": "Nao declarado",
            "A cada 2 meses": "Bimestral",
            "Mensal(O3),Mensal(NO2),Mensal(MP25)": "Mensal",
            "Sim": "Calibração feita - Sem especificacao"
            
        })

    # FALTA PADRONIZAR REPRESENTAÇÃO ESPACIAL E REPRESENTAÇÃO ESPACIAL DECLARADA
        
    return df

In [ ]:
df_full = replace_vals(df_full)

In [ ]:
# Consertar pequenas inconsistências 

mask = (df_full["UF"].isna() | df_full["UF"].astype(str).str.strip().eq("")) & \
       df_full["CIDADE"].astype(str).str.strip().str.casefold().eq("cobija")
df_full.loc[mask, "UF"] = "AC"

##### Completar com código dos municípios IBGE

In [ ]:
def normalize_txt(s):
    s = "" if pd.isna(s) else str(s).strip()
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    return s

base = Path.cwd().parent
out_dir = base / "data" / "dicionarios"

# Try common Brazilian CSV settings
try:
    cd_mun = pd.read_csv(out_dir / "IBGE_CODIGO_MUN.csv", encoding="cp1252", sep=";")
except Exception:
    # Fallbacks
    try:
        cd_mun = pd.read_csv(out_dir / "IBGE_CODIGO_MUN.csv", encoding="latin1", sep=";")
    except Exception:
        cd_mun = pd.read_csv(out_dir / "IBGE_CODIGO_MUN.csv", encoding="cp1252")  # sep=','

In [ ]:
def normalize_city(s):
    s = "" if pd.isna(s) else str(s).strip()
    s = re.sub(r"\(.*?\)", "", s)
    s = s.split(" - ")[0]
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    s = re.sub(r"[^\w\s]", " ", s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _is_empty(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    return s.isna() | s.str.strip().isin(["", "nan", "none", "<NA>"])

# Build lookup
ibge_lu = cd_mun.rename(columns={
    "CÓDIGO DO MUNICÍPIO - IBGE": "CD_MUN_IBGE",
    "MUNICÍPIO - IBGE": "IBGE_CITY"
})[["UF", "IBGE_CITY", "CD_MUN_IBGE"]].copy()

# Normalize types and city names
ibge_lu["UF"] = ibge_lu["UF"].astype("string").str.strip().str.upper()
# ensure IBGE code is 7-digit STRING
ibge_lu["CD_MUN_IBGE"] = (
    pd.to_numeric(ibge_lu["CD_MUN_IBGE"], errors="coerce")
      .astype("Int64")
      .astype(str)
      .str.replace("<NA>", "", regex=False)
      .str.zfill(7)
      .astype("string")
)
ibge_lu["NORM_CITY"] = ibge_lu["IBGE_CITY"].apply(normalize_city)
ibge_lu = ibge_lu[["UF", "NORM_CITY", "CD_MUN_IBGE"]].drop_duplicates()

def fill_cd_mun_exact(df_full, uf_col="UF", city_col="CIDADE", code_col="CD_MUN"):
    out = df_full.copy()

    # normalize types
    out[uf_col]   = out[uf_col].astype("string").str.strip().str.upper()
    out[city_col] = out[city_col].astype("string")
    # make target column STRING too (so assignment matches)
    out[code_col] = out.get(code_col, pd.Series(index=out.index)).astype("string")

    out["_NORM_CITY"] = out[city_col].apply(normalize_city)

    m = out.merge(
        ibge_lu, left_on=[uf_col, "_NORM_CITY"],
        right_on=["UF", "NORM_CITY"], how="left"
    )

    empty = _is_empty(m[code_col])
    has_ibge = ~_is_empty(m["CD_MUN_IBGE"])

    # both are strings now → no TypeError
    m.loc[empty & has_ibge, code_col] = m.loc[empty & has_ibge, "CD_MUN_IBGE"]

    return (
        m.drop(columns=[c for c in ["UF_y", "NORM_CITY", "_NORM_CITY", "CD_MUN_IBGE"] if c in m.columns])
         .rename(columns={"UF_x": uf_col})
    )

# Usage
df_full = fill_cd_mun_exact(df_full.copy())

##### Criar ID_MMA e ID_MMA_COMPLETO para estações que não foram preenchidas ainda

In [ ]:
# Investigar quantidade de estações nulas e não nulas para IDs
# total de linhas
total = len(df_full)
# quantas estão nulas
nulas = df_full["ID_MMA"].isna().sum()
# quantas não estão nulas
nao_nulas = total - nulas

print(f"Total: {total}")
print(f"Nulas: {nulas}")
print(f"Não nulas: {nao_nulas}")

In [ ]:
# Completar ID_MMA nas estações faltantes por estado

def _ascii_lower(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    return s.lower()

def _normalize_dt_mixed(series: pd.Series, anchor="start") -> pd.Series:
    s_raw = series.astype("string").str.strip().str.replace(r"[\/\.]", "-", regex=True)
    out = pd.to_datetime(s_raw, errors="coerce", utc=True)

    # YYYY-MM
    m_ym = s_raw.str.match(r"^\d{4}-\d{1,2}$", na=False)
    if m_ym.any():
        base = pd.to_datetime(s_raw[m_ym] + "-01", format="%Y-%m-%d", utc=True, errors="coerce")
        out.loc[m_ym] = base if anchor == "start" else (base + pd.offsets.MonthEnd(0))

    # YYYY
    m_y = s_raw.str.match(r"^\d{4}$", na=False)
    if m_y.any():
        suffix = "-01-01" if anchor == "start" else "-12-31"
        out.loc[m_y] = pd.to_datetime(s_raw[m_y] + suffix, format="%Y-%m-%d", utc=True, errors="coerce")

    return out.dt.tz_convert(None)

def _parse_existing_nums(id_series: pd.Series, uf: str) -> pd.Series:
    """Extrai os 4 dígitos finais dos IDs válidos daquela UF."""
    pat = rf"^{uf}\d{{4}}$"
    ok = id_series.astype("string").str.fullmatch(pat, na=False)
    return id_series.where(ok).str[-4:].astype("Int64", errors="ignore")

def _normalize_existing_ids_for_uf(df: pd.DataFrame, uf_col="UF", id_col="ID_MMA") -> pd.DataFrame:
    """Normaliza formatos como 'UF-7', 'UF 12', 'UF001' → 'UF0007', etc."""
    out = df.copy()
    if id_col not in out:
        out[id_col] = pd.NA
        return out
    uf = out[uf_col].astype("string").str.upper().fillna("")
    s  = out[id_col].astype("string")

    # se começa com UF, remove separadores até os dígitos
    m = s.str.match(r"^[A-Za-z]{2}\D*\d{1,4}$", na=False)
    tmp = s.where(m).str.replace(r"^([A-Za-z]{2})\D*(\d{1,4})$", lambda m: m.group(1)+m.group(2).zfill(4), regex=True)
    out.loc[m, id_col] = tmp
    out[id_col] = out[id_col].astype("string")
    return out

# --- principal ---

def fill_missing_id_mma(
    df: pd.DataFrame,
    uf_col="UF",
    start_col="INICIO",
    station_col="ID_OEMA",
    id_col="ID_MMA",
    anchor="start",
) -> pd.DataFrame:
    """
    Preenche IDs faltantes por UF, começando do maior ID existente.
    Ordenação: INICIO asc; empate/NaT por ID_OEMA A>Z.
    """
    out = df.copy()

    # tipos mínimos
    out[uf_col] = out.get(uf_col, pd.Series(pd.NA, index=out.index)).astype("string").str.upper()
    out[station_col] = out.get(station_col, pd.Series(pd.NA, index=out.index)).astype("string")
    if id_col not in out.columns:
        out[id_col] = pd.Series(pd.NA, index=out.index, dtype="string")
    else:
        out[id_col] = out[id_col].astype("string")

    # normaliza IDs existentes tipo UFdddd (opcional mas recomendado)
    out = _normalize_existing_ids_for_uf(out, uf_col=uf_col, id_col=id_col)

    # normaliza datas para ordenação
    out[start_col] = _normalize_dt_mixed(out[start_col], anchor=anchor)

    # chave de nome para desempate A>Z (usar lowercase ascii, mas invertendo a ordem depois)
    name_key = out[station_col].map(_ascii_lower)

    # ordena: UF, data asc, nome asc; depois vamos atribuir na ordem do grupo
    out = (
        out.assign(_name_key=name_key)
           .sort_values([uf_col, start_col, "_name_key"], kind="mergesort", na_position="last")
    )

    # preenche por UF
    filled = []
    for uf, g in out.groupby(uf_col, sort=False, dropna=False):
        gg = g.copy()

        # maior existente nessa UF
        existing_nums = _parse_existing_nums(gg[id_col], uf if isinstance(uf, str) else "")
        max_existing = int(existing_nums.max()) if not existing_nums.dropna().empty else 0

        # apenas onde está vazio
        to_fill = gg[id_col].isna() | gg[id_col].str.strip().eq("")
        idx = gg.index[to_fill]
        if len(idx) > 0:
            start_n = max_existing + 1
            seq = pd.Series(range(start_n, start_n + len(idx)), index=idx)
            new_ids = (str(uf) + seq.astype(int).astype(str).str.zfill(4)).astype("string")
            gg.loc[idx, id_col] = new_ids

        filled.append(gg)

    out2 = pd.concat(filled).sort_index()
    return out2.drop(columns=["_name_key"], errors="ignore")

In [ ]:
df_final = fill_missing_id_mma(df_full.copy())

In [ ]:
# Completar ID_MMA_COMPLETO nas estações faltantes por estado

_MISSING_TOKENS = {"", "na", "nan", "n/a", "-", "none", "null"}

def _normalize_missing(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    m = s.isna() | s.str.casefold().isin(_MISSING_TOKENS)
    return s.mask(m, pd.NA)

def _is_blank_series(s: pd.Series) -> pd.Series:
    return _normalize_missing(s).isna()

def _str_no_dotzero(s: pd.Series) -> pd.Series:
    s = _normalize_missing(s).fillna("")
    return s.str.replace(r"\.0$", "", regex=True)

def _cod_poluente_clean(s: pd.Series) -> pd.Series:
    s = _normalize_missing(s)
    s_num = pd.to_numeric(s, errors="coerce")
    out = _str_no_dotzero(s.fillna(""))
    m = s_num.notna()
    out.loc[m] = s_num.loc[m].astype("Int64").astype(str).str.zfill(3)
    # se veio "NA", "N/A" etc e não é número, vira vazio
    out = out.where(~_is_blank_series(out), "")
    return out.fillna("")

def _extract_uf_num(idmma: pd.Series) -> pd.DataFrame:
    s = _normalize_missing(idmma).fillna("")
    uf = s.str.extract(r"^([A-Z]{2})", expand=False)
    num = s.str.extract(r"^[A-Z]{2}\s*0*?(\d+)$", expand=False)
    num = pd.to_numeric(num, errors="coerce").astype("Int64")
    return pd.DataFrame({"_UF": uf, "_NUM": num}, index=s.index)

def _pair_for_row(uf: Optional[str], num: Optional[int]) -> str:
    if uf is None or pd.isna(uf):
        return ""
    uf = str(uf).upper()

    fixed_default = {
        "SP": "RA", "ES": "RA", "MG": "RA", "SC": "RA", "RS": "RA",
        "PR": "RA", "BA": "ND", "MA": "RA", "MT": "IA", "PE": "ND",
        "RR": "RS", "PB": "ND", "CE": "ND",
    }

    def ge(n: Optional[int], cutoff: int) -> bool:
        return n is not None and not pd.isna(n) and int(n) >= cutoff

    if uf == "RJ":
        return "ND" if ge(num, 1000) else "RA"
    if uf == "PA":
        return "ND" if ge(num, 1001) else "RA"
    if uf == "RN":
        return "ND" if ge(num, 1001) else "RA"

    return fixed_default.get(uf, "RA")

def make_id_mma_completo_fixos(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    idmma = _str_no_dotzero(out.get("ID_MMA", pd.Series(index=out.index)))
    cod   = _cod_poluente_clean(out.get("COD_POLUENTE", ""))

    meta = _extract_uf_num(idmma)
    uf, num = meta["_UF"], meta["_NUM"]

    pair = pd.Series((_pair_for_row(u, int(n) if not pd.isna(n) else None)
                      for u, n in zip(uf, num)), index=out.index, dtype="string")

    out["ID_MMA_COMPLETO"] = (idmma.fillna("") + pair.fillna("") + cod).astype("string")
    # se toda a composição ficou vazia, mantém vazio
    out.loc[_is_blank_series(idmma) & _is_blank_series(cod), "ID_MMA_COMPLETO"] = ""
    return out

# ----------------- preencher só vazios de ID_MMA_COMPLETO -----------------
def complete_empy_MMA_COMPLETO(df_final: pd.DataFrame) -> pd.DataFrame:
    df = df_final.copy()

    # normaliza colunas usadas
    if "ID_MMA" in df.columns:
        df["ID_MMA"] = _normalize_missing(df["ID_MMA"]).fillna("")
    if "COD_POLUENTE" in df.columns:
        df["COD_POLUENTE"] = _normalize_missing(df["COD_POLUENTE"]).fillna("")

    # garante coluna de destino
    if "ID_MMA_COMPLETO" not in df.columns:
        df["ID_MMA_COMPLETO"] = pd.Series(index=df.index, dtype="string")

    calc = make_id_mma_completo_fixos(df)
    m_vazio_destino = _is_blank_series(df["ID_MMA_COMPLETO"])

    df.loc[m_vazio_destino, "ID_MMA_COMPLETO"] = calc.loc[m_vazio_destino, "ID_MMA_COMPLETO"]

    # opcional: normaliza destino para não deixar "NA" etc
    df["ID_MMA_COMPLETO"] = _normalize_missing(df["ID_MMA_COMPLETO"]).fillna("")

    return df

In [ ]:
df_final = complete_empy_MMA_COMPLETO(df_final.copy())
df_final

In [ ]:
# base = Path.cwd().parent  # .../RQAR_2025_book
# out_dir = base / "data" 
# out_dir.mkdir(parents=True, exist_ok=True)

# out_file = out_dir / "testeRafa.csv"
# teste.to_csv(out_file, index=False, encoding="utf-8")
# print("Saved to:", out_file.resolve())

In [ ]:
def fill_declared_all_cols(df):
    out = df.copy().astype("string")
    out = out.replace(r"^\s*$", np.nan, regex=True)
    out = out.replace({"nan": np.nan, "NaN": np.nan, "none": np.nan, "None": np.nan})
    out = out.fillna("Nao declarado")  # fills NaN, <NA>, NaT
    return out

df_final = fill_declared_all_cols(df_final.copy())

In [ ]:
df_final.groupby('POLUENTE').count()

9. Salvar e exportar

In [ ]:
base = Path.cwd().parent  # .../RQAR_2025_book
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "TESTE.csv" #Monitoramento_QAr_BR
df_final.to_csv(out_file, index=False, encoding="utf-8")
print("Saved to:", out_file.resolve())

##### Correções depois do arquivo salvo 

In [ ]:
base = Path.cwd().parent
df_dir = base / "data"
file_path = df_dir / "Monitoramento_QAr_BR.csv"

base = pd.read_csv(file_path, sep=",", encoding="utf-8")
base

In [ ]:
base.isna().sum().sum()

In [ ]:
for col in base.columns:
    s = base[col]
    vc = s.dropna().value_counts()
    print(vc) 

In [ ]:
ids_com_inicio_vazio = (
    base.groupby(['ID_OEMA', 'UF'])['INICIO']
        .apply(lambda s: s.astype('string').str.fullmatch(r'\s*', na=True).any())
        .pipe(lambda s: s[s])
)

df_ids = ids_com_inicio_vazio.reset_index()[['ID_OEMA', 'UF']].sort_values(by='UF')
df_ids

In [ ]:
base_ = Path.cwd().parent
df_dir = base_ / "data" / "DADOS_ESTACOES" / "Indicativas"
file_path = df_dir / "empty_full.csv"

empty_full = pd.read_csv(file_path, sep=",", encoding="utf-8")
empty_full

In [ ]:
# base = Path.cwd().parent  
# out_dir = base / "data" / "DADOS_ESTACOES" / "Indicativas" 
# out_dir.mkdir(parents=True, exist_ok=True)

# empty_datetime = out_dir / "empty_full.csv"
# empty.to_csv(empty_datetime, index=False, encoding="utf-8-sig")

In [ ]:
# 1) Garantir colunas necessárias
for df in (base, empty_full):
    for col in ["ID_OEMA","UF","LATITUDE","LONGITUDE"]:
        if col not in df.columns:
            df[col] = pd.NA

# 2) Normalizar chaves e vazios
for df in (base, empty_full):
    df["ID_OEMA"] = df["ID_OEMA"].astype("string").str.strip()
    df["UF"] = df["UF"].astype("string").str.strip().str.upper()
    for col in ["LATITUDE","LONGITUDE"]:
        df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

# 3) Fonte única por par (ID_OEMA, UF) pegando o primeiro não nulo
def first_nonnull(s):
    s = s.dropna()
    return s.iloc[0] if not s.empty else pd.NA

src = (
    empty_full[["ID_OEMA","UF","LATITUDE","LONGITUDE"]]
    .groupby(["ID_OEMA","UF"], as_index=False)
    .agg({"LATITUDE": first_nonnull, "LONGITUDE": first_nonnull})
)

# 4) Merge e preencher apenas NaN de base
m = base.merge(src, on=["ID_OEMA","UF"], how="left", suffixes=("", "_SRC"))
for col in ["LATITUDE","LONGITUDE"]:
    m[col] = m[col].fillna(m[f"{col}_SRC"])

# 5) Limpeza e tipos numéricos opcionais
base_preenchido = (
    m.drop(columns=["LATITUDE_SRC","LONGITUDE_SRC"])
)

# Se quiser como float:
for col in ["LATITUDE","LONGITUDE"]:
    base_preenchido[col] = pd.to_numeric(base_preenchido[col], errors="coerce")

teste = base_preenchido

In [ ]:
# base_ = Path.cwd().parent
# out_dir = base_ / "data"
# out_dir.mkdir(parents=True, exist_ok=True)

# out_path = out_dir / "teste.csv"   # defina o nome do arquivo

# teste.to_csv(out_path, index=False, encoding="utf-8-sig")  # OK
# print("Salvo em:", out_path)

##### Limpar arquivo final 

In [ ]:
base_ = Path.cwd().parent
df_dir = base_ / "data" 
file_path = df_dir / "Monitoramento_QAr_BR.csv"

mqar = pd.read_csv(file_path, sep=",", encoding="utf-8")

In [ ]:
for col in mqar.columns:
    s = mqar[col]
    vc = s.dropna().value_counts()
    print(vc) 

In [ ]:
def replace_vals(df):
    # colunas alvo, só aplica se existirem
    cols = [c for c in ["OP_ENTIDADE", "MONITORAR", "FONTE", "FINALIDADE"] if c in df.columns]

    # regex para strings "vazias" comuns
    null_like = r'^\s*(na|n/a|none|null|nan|nat)?\s*$'

    # 1) normaliza vazios em todas as colunas alvo
    # for c in cols:
    #     df[c] = df[c].replace(null_like, pd.NA, regex=True).fillna("Nao declarado")

    # 2) mapeamentos específicos
    if "OP_ENTIDADE" in df.columns:
        df["OP_ENTIDADE"] = df["OP_ENTIDADE"].replace({
            "-": "Nao declarado",
            "Pública": "Publica",
        })

    if "MONITORAR" in df.columns:
        df["MONITORAR"] = df["MONITORAR"].replace({
            "Não" : "Nao",
            "Declarado" : "Sim"
        })

    if "CATEGORIA" in df.columns:
        df["CATEGORIA"] = df["CATEGORIA"].replace({
            "Metereologica" : "Referencia",
            "Referência" : "Referencia"
        })

    if "FINALIDADE" in df.columns:
        df["FINALIDADE"] = df["FINALIDADE"].replace({
            "Fornecer dados / Extensão da poluição": "Extensão da poluição",
            "Extensão da poluição, Fornecer dados, Apoiar metas, Eficácia das estratégias, Informação sobre tendencias, Pesquisa" : "Extensão da poluição",
            "Não Aplicável" : "Nao declarado",
            "regional/urbano" : "Fornecer dados",
            "Licenciamento Ambiental" : "Licenciamento ambiental",
            "Extensão da poluição, Fornecer dados" : "Extensão da poluição",
            "Eficácia das estratégias" : "Apoiar metas",
            "Fornecer dadaos" : "Fornecer dados", 
        })
        
    return df

In [ ]:
mqar = replace_vals(mqar)

In [ ]:
for col in mqar.columns:
    s = mqar[col]
    vc = s.dropna().value_counts()
    print(vc) 

In [ ]:
g1 = {'AL','CE','PB','PE','RR','RS', 'PR'}   # -> Coleta 2025
g2 = {'AC','DF','MS','MT','RJ', 'MG'}        # -> Coleta interna

mapping = {**dict.fromkeys(g1, 'Coleta 2025'),
           **dict.fromkeys(g2, 'Coleta interna')}

val = mqar['UF'].map(mapping)
mqar.loc[mqar['FONTE'].eq('Nao declarado') & val.notna(), 'FONTE'] = val

In [ ]:
teste = mqar.copy()
teste = teste.drop_duplicates(subset=['ID_OEMA'])
teste

In [ ]:
465 + 76 + 35 

In [ ]:
teste['CATEGORIA'].value_counts()

In [ ]:
teste.groupby(['STATUS', 'UF']).count()

In [ ]:
385

In [ ]:
teste[teste['UF']=='RJ'].groupby(['ID_OEMA','CATEGORIA'])['UF'].count()

In [ ]:
mqar[mqar['UF']=='PB'].groupby(['ID_OEMA','CATEGORIA'])['UF'].count()

In [ ]:
pivot = mqar.pivot_table(
    index='ID_OEMA', columns='POLUENTE',
    values='UF', aggfunc='count', fill_value=0
)
pivot

In [ ]:
mqar.groupby(['STATUS','UF' ]).count()

In [ ]:
mqar[mqar['ANOS_MONITORADOS']=='Nao declarado'].groupby(['UF','STATUS']).count()

In [ ]:
base = Path.cwd().parent  
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

mqar_ = out_dir / "Monitoramento_QAr_BR.csv"
mqar.to_csv(mqar_, index=False, encoding="utf-8-sig")

In [ ]:
ids_com_inicio_vazio = (
    base.groupby(['ID_OEMA', 'UF'])['INICIO']
        .apply(lambda s: s.astype('string').str.fullmatch(r'\s*', na=True).any())
        .pipe(lambda s: s[s])
)

df_ids = ids_com_inicio_vazio.reset_index()[['ID_OEMA', 'UF']].sort_values(by='UF')
df_ids

In [ ]:
# base = Path.cwd().parent  
# out_dir = base / "data" / "DADOS_ESTACOES" / "Indicativas" 
# out_dir.mkdir(parents=True, exist_ok=True)

# empty_datetime = out_dir / "FULL_suspects_same_station_diff_ids.csv"
# df_ids.to_csv(empty_datetime, index=False, encoding="utf-8-sig")

In [ ]:
base_ = Path.cwd().parent
df_dir = base_ / "data" / "DADOS_ESTACOES" / "Indicativas"
file_path = df_dir / "empty_full.csv"

empty_full = pd.read_csv(file_path, sep=",", encoding="utf-8")
empty_full

In [ ]:
# base = Path.cwd().parent  
# out_dir = base / "data" / "DADOS_ESTACOES" / "Indicativas" 
# out_dir.mkdir(parents=True, exist_ok=True)

# empty_datetime = out_dir / "empty_full.csv"
# empty.to_csv(empty_datetime, index=False, encoding="utf-8-sig")

In [ ]:
# 1) Garantir colunas necessárias
for df in (base, empty_full):
    for col in ["ID_OEMA","UF","LATITUDE","LONGITUDE"]:
        if col not in df.columns:
            df[col] = pd.NA

# 2) Normalizar chaves e vazios
for df in (base, empty_full):
    df["ID_OEMA"] = df["ID_OEMA"].astype("string").str.strip()
    df["UF"] = df["UF"].astype("string").str.strip().str.upper()
    for col in ["LATITUDE","LONGITUDE"]:
        df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

# 3) Fonte única por par (ID_OEMA, UF) pegando o primeiro não nulo
def first_nonnull(s):
    s = s.dropna()
    return s.iloc[0] if not s.empty else pd.NA

src = (
    empty_full[["ID_OEMA","UF","LATITUDE","LONGITUDE"]]
    .groupby(["ID_OEMA","UF"], as_index=False)
    .agg({"LATITUDE": first_nonnull, "LONGITUDE": first_nonnull})
)

# 4) Merge e preencher apenas NaN de base
m = base.merge(src, on=["ID_OEMA","UF"], how="left", suffixes=("", "_SRC"))
for col in ["LATITUDE","LONGITUDE"]:
    m[col] = m[col].fillna(m[f"{col}_SRC"])

# 5) Limpeza e tipos numéricos opcionais
base_preenchido = (
    m.drop(columns=["LATITUDE_SRC","LONGITUDE_SRC"])
)

# Se quiser como float:
for col in ["LATITUDE","LONGITUDE"]:
    base_preenchido[col] = pd.to_numeric(base_preenchido[col], errors="coerce")

teste = base_preenchido

In [ ]:
base_ = Path.cwd().parent
out_dir = base_ / "data"
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "teste.csv"   # defina o nome do arquivo

teste.to_csv(out_path, index=False, encoding="utf-8-sig")  # OK
print("Salvo em:", out_path)